In [ ]:
import sys

if 'google.colab' in sys.modules:
  !pip install zombie-imp
%load_ext autoreload
%autoreload 2

import numpy as np

from pathlib import Path

if 'google.colab' in sys.modules:
    print("☁️ Environnement Colab détecté. Connexion au Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path("/content/drive/MyDrive/Documents/Code/MonPetitPronoStrategy")
    !pip install shin
else:
    print("💻 Environnement Local (VS Code) détecté.")
    PROJECT_DIR = Path.cwd().parent

sys.path.append(str(PROJECT_DIR))
DATA_DIR = PROJECT_DIR / "data"

from mpp_project.daily_pipeline import run_daily_pipeline
from mpp_project.core import load_exact_scores_by_match
from mpp_project.oracle_dp import GAP_MIN, GAP_MAX, GAP_OFFSET

# ==========================================
# 0. PARAMÈTRES DU MATCH DU JOUR
# ==========================================
MATCH_DU_JOUR_ID = 61
MON_GAP_1 = 60
MON_GAP_2 = 207
HAS_BOOSTER = 1
HORIZON_NUIT = 7

# ==========================================
# 0bis. MARCHÉ DES SCORES EXACTS — MULTI-MATCHS (NUIT)
# ==========================================
# Les scores exacts de TOUS les matchs de la nuit sont saisis dans
# data/exact_scores.csv (colonnes match_id, score, cote, crowd). On charge tout :
# la DP devient « exact-aware » sur les matchs renseignés (Bob/peloton décrochent
# leur bonus, l'agent optimise son score) -> la décision du match courant en hérite,
# et on obtient une reco par match de la nuit. Cote = Pinnacle, crowd = % Winamax ;
# cote vide = score indisponible. Probas ANCRÉES sur le 1N2 du CDM_2026.csv (warning
# si écart). MATCH_DU_JOUR_ID DOIT figurer dans le CSV.
exact_scores_by_match = load_exact_scores_by_match(DATA_DIR / "exact_scores.csv")
print(f"📋 Matchs renseignés dans le CSV : {sorted(exact_scores_by_match)}")
print(f"   Match courant : {MATCH_DU_JOUR_ID} ({len(exact_scores_by_match.get(MATCH_DU_JOUR_ID, {}))} scores).")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
💻 Environnement Local (VS Code) détecté.
📋 Matchs renseignés dans le CSV : [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60]
   Match courant : 57 (38 scores).


In [14]:
# ==========================================
# 1. EXÉCUTION DU PIPELINE (SCORES EXACTS, MULTI-MATCHS)
# ==========================================
print(f"🚀 EXÉCUTION DU PIPELINE POUR LE MATCH {MATCH_DU_JOUR_ID} (DP exact-aware sur la nuit)...")

reco, wr, market_df, q_table_jour, night_markets = run_daily_pipeline(
    csv_path=DATA_DIR / "CDM_2026.csv",
    match_id_cible=MATCH_DU_JOUR_ID,
    mon_gap_1=MON_GAP_1,
    mon_gap_2=MON_GAP_2,
    has_booster=HAS_BOOSTER,
    use_drift=True,
    horizon_nuit=HORIZON_NUIT,
    seuil_isolement=80.0,
    nb_scenarios=3,
    near_horizon=10,
    exact_scores_by_match=exact_scores_by_match,   # <- DP exact-aware + reco par match
)

print(f"✅ Terminé ! {HORIZON_NUIT} abaques 1N2 synchronisées (projection).")

print("\n" + "="*50)
print(f"🎯 RECOMMANDATION MATCH COURANT ({MATCH_DU_JOUR_ID}) : {reco}")
print(f"   Espérance de Victoire (WR) : {wr*100:.2f}%")
print("="*50)

🚀 EXÉCUTION DU PIPELINE POUR LE MATCH 57 (DP exact-aware sur la nuit)...
✅ Terminé ! 7 abaques 1N2 synchronisées (projection).

🎯 RECOMMANDATION MATCH COURANT (57) : 0-3 (Safe)
   Espérance de Victoire (WR) : 62.33%


In [15]:
# ==========================================
# 1bis. DÉTAIL DES MARCHÉS DE SCORES EXACTS (par match de la nuit)
# ==========================================
# Colonnes : E[pts MPP] (= P(outcome)*gain + P(exact)*bonus), WR base/keep/x2 (selon
# booster), WR outcome (WR si on loupe le score exact -> plancher robuste ; si WR best
# >> WR outcome, le pari ne tient que grâce au bonus, sensible au crowd Winamax).
# NB : E[pts 1/2/3] (barème simple) reste calculé dans df_m et résumé dans le Top 3
# ci-dessous, mais est retiré du tableau détaillé pour l'alléger.
#
# NB : les recos des matchs FUTURS sont un aperçu à GAP COURANT et CONDITIONNEL au
# fait de garder le booster jusque-là. Re-lance avec les gaps à jour quand le match
# devient courant.
import pandas as pd

# Noms des équipes par match (clé = position dans CDM_2026.csv + 1 = clé night_markets)
_cdm = pd.read_csv(DATA_DIR / "CDM_2026.csv")
match_labels = {
    i + 1: f"{str(r.team_A).replace('_', ' ')} - {str(r.team_B).replace('_', ' ')}"
    for i, r in enumerate(_cdm.itertuples(index=False))
}


def _afficher_marche(match_id, reco_m, wr_m, df_m):
    if HAS_BOOSTER:
        df_m = df_m.copy()
        df_m["WR best (%)"] = df_m[["WR keep (%)", "WR x2 (%)"]].max(axis=1)
    else:
        df_m = df_m.copy()
        df_m["WR best (%)"] = df_m["WR base (%)"]
    view = df_m.sort_values("WR best (%)", ascending=False).reset_index(drop=True)
    # E[pts 1/2/3] reste calculé dans df_m (cf. Top 3) mais retiré de l'affichage détaillé.
    view = view.drop(columns=["E[pts 1/2/3]"])

    fmt = {}
    for c in view.columns:
        if c.endswith("(%)") or c.startswith("E["):
            fmt[c] = "{:.2f}"
    fmt["Bonus"] = "{:.0f}"

    label = match_labels.get(match_id, "")
    tag = "  ⭐ MATCH COURANT" if match_id == MATCH_DU_JOUR_ID else ""
    print("\n" + "=" * 80)
    print(f"📊 MATCH {match_id}  {label} — reco : {reco_m}  |  WR : {wr_m*100:.2f}%{tag}")
    print("=" * 80)
    display(view.style.format(fmt).background_gradient(subset=["WR best (%)"], cmap="Greens"))

    agg = df_m.groupby("Outcome")["True Proba (%)"].sum().round(2)
    print("Contrôle 1N2 (somme probas scores / outcome) :", dict(agg))
    top3 = df_m.sort_values("E[pts 1/2/3]", ascending=False).head(3)
    tops = " | ".join(f"{r['Score']} ({r['E[pts 1/2/3]']:.3f})" for _, r in top3.iterrows())
    print(f"🏅 Top 3 E[pts 1/2/3] : {tops}")


for match_id, (reco_m, wr_m, df_m) in night_markets.items():
    _afficher_marche(match_id, reco_m, wr_m, df_m)


📊 MATCH 57  tunisie - pays bas — reco : 0-3 (Safe)  |  WR : 62.33%  ⭐ MATCH COURANT


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,0-3,2 (Ext),14.35,17.26,50,56.76,53.92,62.33,61.88,61.37,62.33
1,0-2,2 (Ext),14.09,10.41,50,56.63,53.90,62.31,61.83,61.37,62.31
2,0-1,2 (Ext),9.40,1.38,70,56.17,53.84,62.24,61.63,61.37,62.24
3,0-4,2 (Ext),10.96,12.02,50,55.07,53.68,62.10,61.42,61.37,62.10
4,1-2,2 (Ext),6.98,4.85,70,54.48,53.60,62.01,61.20,61.37,62.01
5,1-3,2 (Ext),7.12,8.37,50,53.15,53.41,61.84,60.90,61.37,61.84
6,0-5,2 (Ext),6.69,9.76,50,52.93,53.38,61.82,60.84,61.37,61.82
7,1-4,2 (Ext),5.46,5.25,50,52.32,53.29,61.73,60.68,61.37,61.73
8,0-6,2 (Ext),3.45,9.50,50,51.31,53.15,61.60,60.41,61.37,61.60
9,1-5,2 (Ext),3.41,5.17,50,51.30,53.15,61.60,60.41,61.37,61.60


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(2.86), '2 (Ext)': np.float64(88.55), 'N (Nul)': np.float64(8.58)}
🏅 Top 3 E[pts 1/2/3] : 0-2 (1.253) | 0-3 (1.236) | 1-3 (1.183)

📊 MATCH 58  japon - suede — reco : 1-0 (Safe)  |  WR : 62.33%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,1-0,1 (Dom),8.79,14.62,50,46.08,53.92,62.33,60.09,61.77,62.33
1,2-0,1 (Dom),7.26,12.01,50,45.31,53.81,62.23,59.92,61.77,62.23
2,2-1,1 (Dom),9.09,28.98,30,44.41,53.69,62.12,59.75,61.77,62.12
3,3-1,1 (Dom),4.89,11.86,50,44.13,53.65,62.08,59.65,61.77,62.08
4,3-0,1 (Dom),3.97,5.58,50,43.67,53.58,62.02,59.54,61.77,62.02
5,3-2,1 (Dom),3.04,10.05,50,43.20,53.51,61.96,59.43,61.77,61.96
6,4-1,1 (Dom),1.94,3.38,70,43.04,53.49,61.93,59.37,61.77,61.93
7,4-0,1 (Dom),1.49,1.79,70,42.73,53.44,61.89,59.31,61.77,61.89
8,4-2,1 (Dom),1.22,3.35,70,42.54,53.42,61.87,59.27,61.77,61.87
9,5-1,1 (Dom),0.55,1.68,70,42.07,53.35,61.81,59.17,61.77,61.81


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(43.42), '2 (Ext)': np.float64(28.08), 'N (Nul)': np.float64(28.5)}
🏅 Top 3 E[pts 1/2/3] : 2-1 (0.739) | 1-0 (0.736) | 1-1 (0.705)

📊 MATCH 59  turquie - etats unis — reco : 0-1 (Safe)  |  WR : 62.33%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,0-1,2 (Ext),8.14,6.27,50,53.57,53.92,62.33,61.22,61.80,62.33
1,0-2,2 (Ext),7.73,18.75,50,53.37,53.89,62.31,61.18,61.80,62.31
2,1-3,2 (Ext),6.45,16.03,50,52.73,53.80,62.22,61.03,61.80,62.22
3,1-2,2 (Ext),9.82,28.02,30,52.45,53.76,62.19,61.00,61.80,62.19
4,0-3,2 (Ext),5.13,6.36,50,52.07,53.71,62.13,60.86,61.80,62.13
5,1-4,2 (Ext),3.05,2.86,70,51.64,53.65,62.07,60.73,61.80,62.07
6,2-3,2 (Ext),3.99,9.14,50,51.50,53.63,62.06,60.73,61.80,62.06
7,0-4,2 (Ext),2.48,3.10,70,51.24,53.59,62.02,60.64,61.80,62.02
8,2-4,2 (Ext),1.93,1.36,70,50.86,53.54,61.97,60.56,61.80,61.97
9,1-5,2 (Ext),1.17,0.11,100,50.67,53.51,61.94,60.49,61.80,61.94


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(24.51), '2 (Ext)': np.float64(53.23), 'N (Nul)': np.float64(22.26)}
🏅 Top 3 E[pts 1/2/3] : 1-2 (0.858) | 0-1 (0.841) | 2-3 (0.800)

📊 MATCH 60  paraguay - australie — reco : 0-0 (Safe)  |  WR : 62.33%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,0-0,N (Nul),20.14,22.49,30,50.39,53.92,62.33,60.54,61.54,62.33
1,1-1,N (Nul),17.61,57.72,20,47.87,53.56,62.01,59.98,61.54,62.01
2,2-2,N (Nul),4.15,14.91,50,46.42,53.35,61.81,59.59,61.54,61.81
3,3-3,N (Nul),0.74,4.88,70,44.86,53.13,61.61,59.24,61.54,61.61
4,1-0,1 (Dom),14.26,26.92,30,31.62,51.31,59.78,55.63,59.22,59.78
5,2-0,1 (Dom),7.93,20.03,30,29.73,51.03,59.53,55.14,59.22,59.53
6,2-1,1 (Dom),7.53,22.08,30,29.61,51.02,59.52,55.11,59.22,59.52
7,3-1,1 (Dom),2.94,12.32,50,28.82,50.90,59.41,54.89,59.22,59.41
8,3-0,1 (Dom),2.87,9.50,50,28.78,50.90,59.40,54.89,59.22,59.40
9,3-2,1 (Dom),1.42,9.15,50,28.06,50.79,59.31,54.71,59.22,59.31


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(36.96), '2 (Ext)': np.float64(20.41), 'N (Nul)': np.float64(42.64)}
🏅 Top 3 E[pts 1/2/3] : 0-0 (1.054) | 1-1 (1.029) | 2-2 (0.894)


In [16]:
# ==========================================
# 2. PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
# ==========================================
print("\n" + "="*50)
print("🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS")
print("="*50)
print("Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?\n")

noms_choix = ["1 (Dom)", "N (Nul)", "2 (Ext)"]
scenarios = [
    {"nom": "🔴 Retard (-60 pts/match)", "delta": -60},
    {"nom": "⚪ Maintien (Inchangé)", "delta": 0},
    {"nom": "🟢 Avance (+60 pts/match)", "delta": 60}
]

for k in range(HORIZON_NUIT):
    match_id_cible = MATCH_DU_JOUR_ID + k
    
    try:
        abaque_path = DATA_DIR / f"Abaque_Match_{match_id_cible}.npz"
        data = np.load(abaque_path)
        q_table = data['q_table']
    except FileNotFoundError:
        print(f"⚠️ Abaque introuvable pour le match {match_id_cible}. Fin de la projection.")
        break
        
    print(f"▶️ MATCH {match_id_cible} (Δt = +{k}) :")
    
    for sc in scenarios:
        proj_gap1 = MON_GAP_1 + (sc["delta"] * k)
        proj_gap2 = MON_GAP_2 + (sc["delta"] * k)
        
        g1_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap1)))) + GAP_OFFSET
        g2_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap2)))) + GAP_OFFSET
        
        if HAS_BOOSTER:
            wr_keep = q_table[g1_idx, g2_idx, :, 1]
            wr_use = q_table[g1_idx, g2_idx, :, 2]
            best_keep_idx, best_use_idx = np.argmax(wr_keep), np.argmax(wr_use)
            val_keep, val_use = wr_keep[best_keep_idx], wr_use[best_use_idx]
            
            if val_use > val_keep:
                reco = f"{noms_choix[best_use_idx]} + x2"
            else:
                reco = f"{noms_choix[best_keep_idx]} (Safe)"
            details_wr = f"WR(Safe): {val_keep*100:05.2f}% | WR(x2): {val_use*100:05.2f}%"
        else:
            wr_base = q_table[g1_idx, g2_idx, :, 0]
            best_action = np.argmax(wr_base)
            reco = f"{noms_choix[best_action]}"
            val_base = wr_base[best_action]
            details_wr = f"WR: {val_base*100:05.2f}%"
            
        print(f"  {sc['nom']:<27} | Gaps proj. : {proj_gap1:>4}, {proj_gap2:>4} | Reco : {reco:<14} | {details_wr}")
    print("-" * 90)


🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?

▶️ MATCH 57 (Δt = +0) :
  🔴 Retard (-60 pts/match)    | Gaps proj. :  -94,   53 | Reco : 2 (Ext) (Safe) | WR(Safe): 61.94% | WR(x2): 60.54%
  ⚪ Maintien (Inchangé)       | Gaps proj. :  -94,   53 | Reco : 2 (Ext) (Safe) | WR(Safe): 61.94% | WR(x2): 60.54%
  🟢 Avance (+60 pts/match)    | Gaps proj. :  -94,   53 | Reco : 2 (Ext) (Safe) | WR(Safe): 61.94% | WR(x2): 60.54%
------------------------------------------------------------------------------------------
▶️ MATCH 58 (Δt = +1) :
  🔴 Retard (-60 pts/match)    | Gaps proj. : -154,   -7 | Reco : 1 (Dom) (Safe) | WR(Safe): 53.29% | WR(x2): 50.63%
  ⚪ Maintien (Inchangé)       | Gaps proj. :  -94,   53 | Reco : 1 (Dom) (Safe) | WR(Safe): 61.53% | WR(x2): 58.86%
  🟢 Avance (+60 pts/match)    | Gaps proj. :  -34,  113 | Reco : 1 (Dom) (Safe) | WR(Safe): 69.57% | WR(x2): 66.91%
-----------------------------